# Germany and EU Tourism Competitiveness Analysis
## 02 - Data Cleaning

### Purpose

This notebook prepares the raw Eurostat tourism and economic datasets for reliable statistical and business analysis.

The cleaning process focuses on data integrity rather than altering the underlying official observations.

The main objectives are to:

- inspect the structure and quality of each raw dataset
- standardize column names and data types
- validate country, period, and source-market identifiers
- identify duplicate records
- preserve genuine missing observations
- retain Eurostat confidentiality and provisional status information
- distinguish missing values from zero values
- assess period completeness before aggregation
- separate individual source markets from geographic aggregates
- create consistent analysis-ready datasets
- document all transformations from raw to clean data

Raw files in `data/raw/` will remain unchanged.

Cleaned datasets will be saved separately in `data/clean/`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
project_root = Path.cwd().parent

raw_path = project_root / "data" / "raw"
clean_path = project_root / "data" / "clean"

clean_path.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", project_root)
print("Raw data:", raw_path)
print("Clean data:", clean_path)

Project root: /Users/jannoelvero/Documents/germany_eu_tourism_analysis
Raw data: /Users/jannoelvero/Documents/germany_eu_tourism_analysis/data/raw
Clean data: /Users/jannoelvero/Documents/germany_eu_tourism_analysis/data/clean


In [3]:
germany_nights_raw = pd.read_csv(
    raw_path /
    "eurostat_germany_foreign_nights_monthly_raw.csv"
)

eu27_nights_raw = pd.read_csv(
    raw_path /
    "eurostat_eu27_foreign_nights_monthly_raw.csv"
)

germany_arrivals_raw = pd.read_csv(
    raw_path /
    "eurostat_germany_foreign_arrivals_monthly_raw.csv"
)

eu27_arrivals_raw = pd.read_csv(
    raw_path /
    "eurostat_eu27_foreign_arrivals_monthly_raw.csv"
)

source_nights_raw = pd.read_csv(
    raw_path /
    "eurostat_germany_source_market_nights_annual_raw.csv"
)

source_arrivals_raw = pd.read_csv(
    raw_path /
    "eurostat_germany_source_market_arrivals_annual_raw.csv"
)

travel_bop_raw = pd.read_csv(
    raw_path /
    "eurostat_germany_travel_bop_annual_raw.csv"
)

eu27_reference = pd.read_csv(
    raw_path /
    "eu27_country_reference.csv"
)

collection_metadata = pd.read_csv(
    raw_path /
    "data_collection_metadata.csv"
)

print("All raw datasets loaded successfully.")

All raw datasets loaded successfully.


In [4]:
datasets = {
    "Germany Nights": germany_nights_raw,
    "EU27 Nights": eu27_nights_raw,
    "Germany Arrivals": germany_arrivals_raw,
    "EU27 Arrivals": eu27_arrivals_raw,
    "Source Market Nights": source_nights_raw,
    "Source Market Arrivals": source_arrivals_raw,
    "Travel BoP": travel_bop_raw,
    "EU27 Reference": eu27_reference,
    "Collection Metadata": collection_metadata
}

structure_summary = []

for name, df in datasets.items():

    structure_summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum()
    })

structure_summary = pd.DataFrame(
    structure_summary
)

structure_summary

,dataset,rows,columns,duplicate_rows,missing_values
0,Germany Nights,67,7,0,68
1,EU27 Nights,1809,7,0,1793
2,Germany Arrivals,67,7,0,68
3,EU27 Arrivals,1809,7,0,1800
4,Source Market Nights,315,7,0,382
5,Source Market Arrivals,315,7,0,382
6,Travel BoP,30,8,0,24
7,EU27 Reference,27,2,0,0
8,Collection Metadata,7,12,0,0


In [5]:
for name, df in datasets.items():

    print(f"\n{'=' * 60}")
    print(name)
    print("=" * 60)

    print(df.dtypes)


Germany Nights
country_code          str
country               str
period                str
foreign_nights    float64
status            float64
year                int64
month               int64
dtype: object

EU27 Nights
country_code          str
country               str
period                str
foreign_nights    float64
status                str
year                int64
month               int64
dtype: object

Germany Arrivals
country_code            str
country                 str
period                  str
foreign_arrivals    float64
status              float64
year                  int64
month                 int64
dtype: object

EU27 Arrivals
country_code            str
country                 str
period                  str
foreign_arrivals    float64
status                  str
year                  int64
month                 int64
dtype: object

Source Market Nights
destination_code          str
destination               str
source_market_code        str
source_market 

## Initial Structural Audit

The initial audit identified no fully duplicated rows in any raw dataset.

The raw datasets have generally appropriate structures and numeric data types. However, the initial total missing-value counts require further investigation because Eurostat status columns are frequently empty when no observation-level flag applies.

An empty status field does not mean that the tourism observation itself is missing.

Therefore, missingness will be evaluated separately for:

- analytical numeric values
- Eurostat status fields
- country and market identifiers
- time variables

No rows or missing values will be removed at this stage.

In [6]:
missing_summary = []

for name, df in datasets.items():

    for column in df.columns:

        missing_count = df[column].isna().sum()

        missing_summary.append({
            "dataset": name,
            "column": column,
            "missing_count": missing_count,
            "missing_pct": round(
                missing_count / len(df) * 100,
                2
            )
        })

missing_summary = pd.DataFrame(
    missing_summary
)

missing_summary[
    missing_summary["missing_count"] > 0
]

,dataset,column,missing_count,missing_pct
3,Germany Nights,foreign_nights,1,1.49
4,Germany Nights,status,67,100.00
10,EU27 Nights,foreign_nights,51,2.82
11,EU27 Nights,status,1742,96.30
17,Germany Arrivals,foreign_arrivals,1,1.49
18,Germany Arrivals,status,67,100.00
24,EU27 Arrivals,foreign_arrivals,57,3.15
25,EU27 Arrivals,status,1743,96.35
33,Source Market Nights,foreign_nights,67,21.27
34,Source Market Nights,status,315,100.00


In [7]:
status_datasets = {
    "Germany Nights": germany_nights_raw,
    "EU27 Nights": eu27_nights_raw,
    "Germany Arrivals": germany_arrivals_raw,
    "EU27 Arrivals": eu27_arrivals_raw,
    "Source Market Nights": source_nights_raw,
    "Source Market Arrivals": source_arrivals_raw,
    "Travel BoP": travel_bop_raw
}

for name, df in status_datasets.items():

    print(f"\n{name}")

    print(
        df["status"]
        .value_counts(
            dropna=False
        )
    )


Germany Nights
status
NaN    67
Name: count, dtype: int64

EU27 Nights
status
NaN    1742
e        39
|C       26
u         2
Name: count, dtype: int64

Germany Arrivals
status
NaN    67
Name: count, dtype: int64

EU27 Arrivals
status
NaN    1743
e        39
|C       25
u         2
Name: count, dtype: int64

Source Market Nights
status
NaN    315
Name: count, dtype: int64

Source Market Arrivals
status
NaN    315
Name: count, dtype: int64

Travel BoP
status
NaN    24
p       6
Name: count, dtype: int64


In [8]:
measure_columns = {
    "Germany Nights": (
        germany_nights_raw,
        "foreign_nights"
    ),
    "EU27 Nights": (
        eu27_nights_raw,
        "foreign_nights"
    ),
    "Germany Arrivals": (
        germany_arrivals_raw,
        "foreign_arrivals"
    ),
    "EU27 Arrivals": (
        eu27_arrivals_raw,
        "foreign_arrivals"
    ),
    "Source Market Nights": (
        source_nights_raw,
        "foreign_nights"
    ),
    "Source Market Arrivals": (
        source_arrivals_raw,
        "foreign_arrivals"
    ),
    "Travel BoP": (
        travel_bop_raw,
        "travel_mio_eur"
    )
}

measure_missing_summary = []

for name, (
    df,
    measure
) in measure_columns.items():

    missing_count = (
        df[measure]
        .isna()
        .sum()
    )

    measure_missing_summary.append({
        "dataset": name,
        "measure": measure,
        "rows": len(df),
        "missing_measure_values":
            missing_count,
        "missing_pct": round(
            missing_count /
            len(df) * 100,
            2
        )
    })

measure_missing_summary = pd.DataFrame(
    measure_missing_summary
)

measure_missing_summary

,dataset,measure,rows,missing_measure_values,missing_pct
0,Germany Nights,foreign_nights,67,1,1.49
1,EU27 Nights,foreign_nights,1809,51,2.82
2,Germany Arrivals,foreign_arrivals,67,1,1.49
3,EU27 Arrivals,foreign_arrivals,1809,57,3.15
4,Source Market Nights,foreign_nights,315,67,21.27
5,Source Market Arrivals,foreign_arrivals,315,67,21.27
6,Travel BoP,travel_mio_eur,30,0,0.00


In [9]:
identifier_checks = {
    "Germany Nights": [
        "country_code",
        "period",
        "year",
        "month"
    ],
    "EU27 Nights": [
        "country_code",
        "period",
        "year",
        "month"
    ],
    "Germany Arrivals": [
        "country_code",
        "period",
        "year",
        "month"
    ],
    "EU27 Arrivals": [
        "country_code",
        "period",
        "year",
        "month"
    ],
    "Source Market Nights": [
        "destination_code",
        "source_market_code",
        "year"
    ],
    "Source Market Arrivals": [
        "destination_code",
        "source_market_code",
        "year"
    ],
    "Travel BoP": [
        "country_code",
        "partner_code",
        "flow_code",
        "year"
    ]
}

identifier_summary = []

for name, columns in identifier_checks.items():

    df = datasets[name]

    for column in columns:

        identifier_summary.append({
            "dataset": name,
            "identifier": column,
            "missing": df[column].isna().sum()
        })

identifier_summary = pd.DataFrame(
    identifier_summary
)

identifier_summary

,dataset,identifier,missing
0,Germany Nights,country_code,0
1,Germany Nights,period,0
2,Germany Nights,year,0
3,Germany Nights,month,0
4,EU27 Nights,country_code,0
5,EU27 Nights,period,0
6,EU27 Nights,year,0
7,EU27 Nights,month,0
8,Germany Arrivals,country_code,0
9,Germany Arrivals,period,0


## Cleaning and Standardization

The raw datasets have passed the initial structural audit.

Cleaning will now be performed on separate working copies so that the original imported raw datasets remain unchanged within the notebook.

The cleaning process will preserve Eurostat observations and status information. Missing official observations will not be replaced with zero or statistically imputed.

In [10]:
germany_nights_clean = germany_nights_raw.copy()
eu27_nights_clean = eu27_nights_raw.copy()

germany_arrivals_clean = germany_arrivals_raw.copy()
eu27_arrivals_clean = eu27_arrivals_raw.copy()

source_nights_clean = source_nights_raw.copy()
source_arrivals_clean = source_arrivals_raw.copy()

travel_bop_clean = travel_bop_raw.copy()

eu27_reference_clean = eu27_reference.copy()

print("Clean working copies created.")

Clean working copies created.


In [11]:
status_clean_datasets = [
    germany_nights_clean,
    eu27_nights_clean,
    germany_arrivals_clean,
    eu27_arrivals_clean,
    source_nights_clean,
    source_arrivals_clean,
    travel_bop_clean
]

for df in status_clean_datasets:
    df["status"] = df["status"].astype("string")

print("Status columns standardized.")

Status columns standardized.


In [12]:
for name, df in {
    "Germany Nights": germany_nights_clean,
    "EU27 Nights": eu27_nights_clean,
    "Germany Arrivals": germany_arrivals_clean,
    "EU27 Arrivals": eu27_arrivals_clean,
    "Source Market Nights": source_nights_clean,
    "Source Market Arrivals": source_arrivals_clean,
    "Travel BoP": travel_bop_clean
}.items():

    print(
        name,
        "| dtype:",
        df["status"].dtype,
        "| flags:",
        df["status"].dropna().unique().tolist()
    )

Germany Nights | dtype: string | flags: []
EU27 Nights | dtype: string | flags: ['u', '|C', 'e']
Germany Arrivals | dtype: string | flags: []
EU27 Arrivals | dtype: string | flags: ['u', '|C', 'e']
Source Market Nights | dtype: string | flags: []
Source Market Arrivals | dtype: string | flags: []
Travel BoP | dtype: string | flags: ['p']


In [13]:
germany_nights_clean["value_available"] = (
    germany_nights_clean[
        "foreign_nights"
    ].notna()
)

eu27_nights_clean["value_available"] = (
    eu27_nights_clean[
        "foreign_nights"
    ].notna()
)

germany_arrivals_clean["value_available"] = (
    germany_arrivals_clean[
        "foreign_arrivals"
    ].notna()
)

eu27_arrivals_clean["value_available"] = (
    eu27_arrivals_clean[
        "foreign_arrivals"
    ].notna()
)

In [14]:
source_nights_clean["value_available"] = (
    source_nights_clean[
        "foreign_nights"
    ].notna()
)

source_arrivals_clean["value_available"] = (
    source_arrivals_clean[
        "foreign_arrivals"
    ].notna()
)

In [15]:
travel_bop_clean["value_available"] = (
    travel_bop_clean[
        "travel_mio_eur"
    ].notna()
)

In [16]:
availability_summary = pd.DataFrame({
    "dataset": [
        "Germany Nights",
        "EU27 Nights",
        "Germany Arrivals",
        "EU27 Arrivals",
        "Source Market Nights",
        "Source Market Arrivals",
        "Travel BoP"
    ],
    "available": [
        germany_nights_clean["value_available"].sum(),
        eu27_nights_clean["value_available"].sum(),
        germany_arrivals_clean["value_available"].sum(),
        eu27_arrivals_clean["value_available"].sum(),
        source_nights_clean["value_available"].sum(),
        source_arrivals_clean["value_available"].sum(),
        travel_bop_clean["value_available"].sum()
    ],
    "unavailable": [
        (~germany_nights_clean["value_available"]).sum(),
        (~eu27_nights_clean["value_available"]).sum(),
        (~germany_arrivals_clean["value_available"]).sum(),
        (~eu27_arrivals_clean["value_available"]).sum(),
        (~source_nights_clean["value_available"]).sum(),
        (~source_arrivals_clean["value_available"]).sum(),
        (~travel_bop_clean["value_available"]).sum()
    ]
})

availability_summary

,dataset,available,unavailable
0,Germany Nights,66,1
1,EU27 Nights,1758,51
2,Germany Arrivals,66,1
3,EU27 Arrivals,1752,57
4,Source Market Nights,248,67
5,Source Market Arrivals,248,67
6,Travel BoP,30,0


## Eurostat Observation Quality Flags

Eurostat status information will be preserved rather than removed.

The observed status codes have different meanings:

- `e` indicates an estimated observation
- `u` indicates an observation with low reliability
- `p` indicates a provisional observation
- `|C` indicates confidentiality in the extracted Eurostat response

These flags do not all imply missing data.

Estimated, low-reliability, and provisional observations may contain published numeric values. Confidential observations may instead contain no publicly disclosed value.

The relationship between status flags and numeric availability will therefore be inspected before quality indicators are created.

In [17]:
eu27_nights_flag_check = (
    eu27_nights_clean
    .groupby(
        ["status", "value_available"],
        dropna=False
    )
    .size()
    .reset_index(
        name="observations"
    )
)

eu27_nights_flag_check

,status,value_available,observations
0,e,True,39
1,u,True,2
2,|C,False,26
3,<NA>,False,25
4,<NA>,True,1717


In [18]:
eu27_arrivals_flag_check = (
    eu27_arrivals_clean
    .groupby(
        ["status", "value_available"],
        dropna=False
    )
    .size()
    .reset_index(
        name="observations"
    )
)

eu27_arrivals_flag_check

,status,value_available,observations
0,e,True,39
1,u,True,2
2,|C,False,25
3,<NA>,False,32
4,<NA>,True,1711


In [19]:
travel_bop_flag_check = (
    travel_bop_clean
    .groupby(
        ["status", "value_available"],
        dropna=False
    )
    .size()
    .reset_index(
        name="observations"
    )
)

travel_bop_flag_check

,status,value_available,observations
0,p,True,6
1,<NA>,True,24


In [20]:
eu27_nights_flagged = (
    eu27_nights_clean[
        eu27_nights_clean[
            "status"
        ].notna()
    ][
        [
            "country_code",
            "country",
            "period",
            "foreign_nights",
            "status",
            "value_available"
        ]
    ]
    .sort_values(
        [
            "country",
            "period"
        ]
    )
)

eu27_nights_flagged

,country_code,country,period,foreign_nights,status,value_available
131,BE,Belgium,2026-05,2274371.0,u,True
132,BE,Belgium,2026-06,1997487.0,u,True
134,BG,Bulgaria,2021-01,NaN,|C,False
135,BG,Bulgaria,2021-02,NaN,|C,False
136,BG,Bulgaria,2021-03,NaN,|C,False
137,BG,Bulgaria,2021-04,NaN,|C,False
138,BG,Bulgaria,2021-05,NaN,|C,False
143,BG,Bulgaria,2021-10,NaN,|C,False
146,BG,Bulgaria,2022-01,NaN,|C,False
147,BG,Bulgaria,2022-02,NaN,|C,False


In [21]:
eu27_arrivals_flagged = (
    eu27_arrivals_clean[
        eu27_arrivals_clean[
            "status"
        ].notna()
    ][
        [
            "country_code",
            "country",
            "period",
            "foreign_arrivals",
            "status",
            "value_available"
        ]
    ]
    .sort_values(
        [
            "country",
            "period"
        ]
    )
)

eu27_arrivals_flagged

,country_code,country,period,foreign_arrivals,status,value_available
131,BE,Belgium,2026-05,990747.0,u,True
132,BE,Belgium,2026-06,869194.0,u,True
134,BG,Bulgaria,2021-01,NaN,|C,False
135,BG,Bulgaria,2021-02,NaN,|C,False
136,BG,Bulgaria,2021-03,NaN,|C,False
137,BG,Bulgaria,2021-04,NaN,|C,False
138,BG,Bulgaria,2021-05,NaN,|C,False
143,BG,Bulgaria,2021-10,NaN,|C,False
146,BG,Bulgaria,2022-01,NaN,|C,False
147,BG,Bulgaria,2022-02,NaN,|C,False


In [22]:
nights_flag_summary = (
    eu27_nights_flagged
    .groupby(
        [
            "country",
            "status"
        ]
    )
    .agg(
        observations=(
            "period",
            "count"
        ),
        first_period=(
            "period",
            "min"
        ),
        last_period=(
            "period",
            "max"
        ),
        values_available=(
            "value_available",
            "sum"
        )
    )
    .reset_index()
)

nights_flag_summary

,country,status,observations,first_period,last_period,values_available
0,Belgium,u,2,2026-05,2026-06,2
1,Bulgaria,|C,19,2021-01,2023-12,0
2,Estonia,|C,7,2021-11,2023-11,0
3,France,e,3,2023-10,2023-12,3
4,Ireland,e,36,2021-01,2023-12,36


In [23]:
arrivals_flag_summary = (
    eu27_arrivals_flagged
    .groupby(
        [
            "country",
            "status"
        ]
    )
    .agg(
        observations=(
            "period",
            "count"
        ),
        first_period=(
            "period",
            "min"
        ),
        last_period=(
            "period",
            "max"
        ),
        values_available=(
            "value_available",
            "sum"
        )
    )
    .reset_index()
)

arrivals_flag_summary

,country,status,observations,first_period,last_period,values_available
0,Belgium,u,2,2026-05,2026-06,2
1,Bulgaria,|C,17,2021-01,2023-12,0
2,Estonia,|C,7,2021-11,2023-11,0
3,France,e,3,2023-10,2023-12,3
4,France,|C,1,2021-06,2021-06,0
5,Ireland,e,36,2021-01,2023-12,36


In [24]:
eu27_nights_clean[
    "is_estimated"
] = (
    eu27_nights_clean["status"] == "e"
)

eu27_nights_clean[
    "is_low_reliability"
] = (
    eu27_nights_clean["status"] == "u"
)

eu27_nights_clean[
    "is_confidential"
] = (
    eu27_nights_clean["status"]
    .str.contains(
        "C",
        na=False
    )
)

In [25]:
eu27_arrivals_clean[
    "is_estimated"
] = (
    eu27_arrivals_clean["status"] == "e"
)

eu27_arrivals_clean[
    "is_low_reliability"
] = (
    eu27_arrivals_clean["status"] == "u"
)

eu27_arrivals_clean[
    "is_confidential"
] = (
    eu27_arrivals_clean["status"]
    .str.contains(
        "C",
        na=False
    )
)

In [26]:
travel_bop_clean[
    "is_provisional"
] = (
    travel_bop_clean["status"] == "p"
)

### Observation Quality Assessment

The Eurostat quality flags behave differently and therefore require separate treatment.

- Estimated (`e`) observations contain published numeric values and will remain available for analysis.
- Low-reliability (`u`) observations also contain published numeric values and will remain available, but results involving them will be interpreted cautiously.
- Confidential (`|C`) observations do not contain publicly disclosed numeric values and will remain missing.
- Provisional (`p`) Balance of Payments observations contain numeric values and will remain available while being clearly identified as provisional.

No flagged observation will be automatically deleted.

Quality flags will be retained alongside the original Eurostat `status` field so that later analytical stages can apply appropriate completeness and reliability rules.

In [27]:
quality_flag_summary = pd.DataFrame({
    "dataset": [
        "EU27 Nights",
        "EU27 Arrivals",
        "Travel BoP"
    ],
    "estimated": [
        eu27_nights_clean["is_estimated"].sum(),
        eu27_arrivals_clean["is_estimated"].sum(),
        0
    ],
    "low_reliability": [
        eu27_nights_clean["is_low_reliability"].sum(),
        eu27_arrivals_clean["is_low_reliability"].sum(),
        0
    ],
    "confidential": [
        eu27_nights_clean["is_confidential"].sum(),
        eu27_arrivals_clean["is_confidential"].sum(),
        0
    ],
    "provisional": [
        0,
        0,
        travel_bop_clean["is_provisional"].sum()
    ]
})

quality_flag_summary

,dataset,estimated,low_reliability,confidential,provisional
0,EU27 Nights,39,2,26,0
1,EU27 Arrivals,39,2,25,0
2,Travel BoP,0,0,0,6


In [28]:
monthly_clean_datasets = [
    germany_nights_clean,
    eu27_nights_clean,
    germany_arrivals_clean,
    eu27_arrivals_clean
]

for df in monthly_clean_datasets:

    df["date"] = pd.to_datetime(
        df["period"] + "-01"
    )

print("Monthly date fields created.")

Monthly date fields created.


In [29]:
time_validation = []

for name, df in {
    "Germany Nights": germany_nights_clean,
    "EU27 Nights": eu27_nights_clean,
    "Germany Arrivals": germany_arrivals_clean,
    "EU27 Arrivals": eu27_arrivals_clean
}.items():

    year_mismatch = (
        df["year"]
        != df["date"].dt.year
    ).sum()

    month_mismatch = (
        df["month"]
        != df["date"].dt.month
    ).sum()

    time_validation.append({
        "dataset": name,
        "year_mismatch": year_mismatch,
        "month_mismatch": month_mismatch
    })

time_validation = pd.DataFrame(
    time_validation
)

time_validation

,dataset,year_mismatch,month_mismatch
0,Germany Nights,0,0
1,EU27 Nights,0,0
2,Germany Arrivals,0,0
3,EU27 Arrivals,0,0


In [30]:
nights_completeness = (
    eu27_nights_clean
    .groupby(
        ["country_code", "country", "year"]
    )
    .agg(
        months_listed=(
            "month",
            "count"
        ),
        months_available=(
            "value_available",
            "sum"
        )
    )
    .reset_index()
)

nights_completeness[
    "complete_12_month_year"
] = (
    nights_completeness[
        "months_available"
    ] == 12
)

nights_completeness

,country_code,country,year,months_listed,months_available,complete_12_month_year
0,AT,Austria,2021,12,12,True
1,AT,Austria,2022,12,12,True
2,AT,Austria,2023,12,12,True
3,AT,Austria,2024,12,12,True
4,AT,Austria,2025,12,12,True
...,...,...,...,...,...,...
157,SK,Slovakia,2022,12,12,True
158,SK,Slovakia,2023,12,12,True
159,SK,Slovakia,2024,12,12,True
160,SK,Slovakia,2025,12,12,True


In [31]:
arrivals_completeness = (
    eu27_arrivals_clean
    .groupby(
        ["country_code", "country", "year"]
    )
    .agg(
        months_listed=(
            "month",
            "count"
        ),
        months_available=(
            "value_available",
            "sum"
        )
    )
    .reset_index()
)

arrivals_completeness[
    "complete_12_month_year"
] = (
    arrivals_completeness[
        "months_available"
    ] == 12
)

arrivals_completeness

,country_code,country,year,months_listed,months_available,complete_12_month_year
0,AT,Austria,2021,12,12,True
1,AT,Austria,2022,12,12,True
2,AT,Austria,2023,12,12,True
3,AT,Austria,2024,12,12,True
4,AT,Austria,2025,12,12,True
...,...,...,...,...,...,...
157,SK,Slovakia,2022,12,12,True
158,SK,Slovakia,2023,12,12,True
159,SK,Slovakia,2024,12,12,True
160,SK,Slovakia,2025,12,12,True


In [32]:
incomplete_nights_history = (
    nights_completeness[
        (nights_completeness["year"] <= 2025)
        &
        (~nights_completeness[
            "complete_12_month_year"
        ])
    ]
)

incomplete_nights_history

,country_code,country,year,months_listed,months_available,complete_12_month_year
12,BG,Bulgaria,2021,12,6,False
13,BG,Bulgaria,2022,12,6,False
14,BG,Bulgaria,2023,12,5,False
42,EE,Estonia,2021,12,11,False
43,EE,Estonia,2022,12,9,False
44,EE,Estonia,2023,12,9,False


In [33]:
incomplete_arrivals_history = (
    arrivals_completeness[
        (arrivals_completeness["year"] <= 2025)
        &
        (~arrivals_completeness[
            "complete_12_month_year"
        ])
    ]
)

incomplete_arrivals_history

,country_code,country,year,months_listed,months_available,complete_12_month_year
6,BE,Belgium,2021,12,11,False
12,BG,Bulgaria,2021,12,6,False
13,BG,Bulgaria,2022,12,6,False
14,BG,Bulgaria,2023,12,7,False
16,BG,Bulgaria,2025,12,11,False
36,DK,Denmark,2021,12,11,False
37,DK,Denmark,2022,12,11,False
42,EE,Estonia,2021,12,11,False
43,EE,Estonia,2022,12,9,False
44,EE,Estonia,2023,12,9,False


### Annual Completeness Rules

Monthly observations will be evaluated for annual completeness before annual totals are calculated.

A historical country-year is considered complete when all 12 monthly numeric observations are available.

Incomplete country-years will remain in the clean dataset, but they will be explicitly flagged and will not be treated as complete annual totals in analyses requiring full-year comparability.

The year 2026 is treated separately because it represents an intentionally incomplete year-to-date reporting period rather than a historical full year.

In [34]:
nights_complete_lookup = (
    nights_completeness[
        [
            "country_code",
            "year",
            "months_available",
            "complete_12_month_year"
        ]
    ]
)

eu27_nights_clean = eu27_nights_clean.merge(
    nights_complete_lookup,
    on=[
        "country_code",
        "year"
    ],
    how="left"
)

In [35]:
arrivals_complete_lookup = (
    arrivals_completeness[
        [
            "country_code",
            "year",
            "months_available",
            "complete_12_month_year"
        ]
    ]
)

eu27_arrivals_clean = eu27_arrivals_clean.merge(
    arrivals_complete_lookup,
    on=[
        "country_code",
        "year"
    ],
    how="left"
)

In [36]:
for df in [
    germany_nights_clean,
    germany_arrivals_clean
]:

    df["months_available"] = (
        df.groupby("year")[
            "value_available"
        ]
        .transform("sum")
    )

    df["complete_12_month_year"] = (
        df["months_available"] == 12
    )

In [37]:
print(
    "Germany Nights:"
)

print(
    germany_nights_clean
    .groupby("year")
    [
        [
            "months_available",
            "complete_12_month_year"
        ]
    ]
    .first()
)

print(
    "\nGermany Arrivals:"
)

print(
    germany_arrivals_clean
    .groupby("year")
    [
        [
            "months_available",
            "complete_12_month_year"
        ]
    ]
    .first()
)

Germany Nights:
      months_available  complete_12_month_year
year                                          
2021                12                    True
2022                12                    True
2023                12                    True
2024                12                    True
2025                12                    True
2026                 6                   False

Germany Arrivals:
      months_available  complete_12_month_year
year                                          
2021                12                    True
2022                12                    True
2023                12                    True
2024                12                    True
2025                12                    True
2026                 6                   False


In [38]:
nights_ytd_2026 = (
    eu27_nights_clean[
        (eu27_nights_clean["year"] == 2026)
        &
        (eu27_nights_clean["month"] <= 6)
    ]
    .groupby(
        ["country_code", "country"]
    )
    .agg(
        ytd_months_available=(
            "value_available",
            "sum"
        )
    )
    .reset_index()
)

nights_ytd_2026[
    "complete_jan_jun_2026"
] = (
    nights_ytd_2026[
        "ytd_months_available"
    ] == 6
)

nights_ytd_2026

,country_code,country,ytd_months_available,complete_jan_jun_2026
0,AT,Austria,6,True
1,BE,Belgium,6,True
2,BG,Bulgaria,6,True
3,CY,Cyprus,6,True
4,CZ,Czechia,6,True
5,DE,Germany,6,True
6,DK,Denmark,6,True
7,EE,Estonia,6,True
8,EL,Greece,6,True
9,ES,Spain,6,True


In [39]:
arrivals_ytd_2026 = (
    eu27_arrivals_clean[
        (eu27_arrivals_clean["year"] == 2026)
        &
        (eu27_arrivals_clean["month"] <= 6)
    ]
    .groupby(
        ["country_code", "country"]
    )
    .agg(
        ytd_months_available=(
            "value_available",
            "sum"
        )
    )
    .reset_index()
)

arrivals_ytd_2026[
    "complete_jan_jun_2026"
] = (
    arrivals_ytd_2026[
        "ytd_months_available"
    ] == 6
)

arrivals_ytd_2026

,country_code,country,ytd_months_available,complete_jan_jun_2026
0,AT,Austria,6,True
1,BE,Belgium,6,True
2,BG,Bulgaria,6,True
3,CY,Cyprus,6,True
4,CZ,Czechia,6,True
5,DE,Germany,6,True
6,DK,Denmark,6,True
7,EE,Estonia,6,True
8,EL,Greece,6,True
9,ES,Spain,6,True


In [40]:
print(
    "Nights countries with complete "
    "Jan-Jun 2026:",
    nights_ytd_2026[
        "complete_jan_jun_2026"
    ].sum()
)

print(
    "Arrivals countries with complete "
    "Jan-Jun 2026:",
    arrivals_ytd_2026[
        "complete_jan_jun_2026"
    ].sum()
)

Nights countries with complete Jan-Jun 2026: 27
Arrivals countries with complete Jan-Jun 2026: 26


In [41]:
print("Incomplete Nights:")
display(
    nights_ytd_2026[
        ~nights_ytd_2026[
            "complete_jan_jun_2026"
        ]
    ]
)

print("Incomplete Arrivals:")
display(
    arrivals_ytd_2026[
        ~arrivals_ytd_2026[
            "complete_jan_jun_2026"
        ]
    ]
)

Incomplete Nights:


,country_code,country,ytd_months_available,complete_jan_jun_2026


Incomplete Arrivals:


,country_code,country,ytd_months_available,complete_jan_jun_2026
22,PT,Portugal,4,False


In [42]:
source_market_reference = (
    source_nights_clean[
        [
            "source_market_code",
            "source_market"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "source_market_code"
    )
    .reset_index(
        drop=True
    )
)

source_market_reference

,source_market_code,source_market
0,AFR,Africa
1,AFR_OTH,Other African countries (aggregate changing ac...
2,AME,America
3,AME_C_S,Central and South America
4,AME_C_S_OTH,Other Central or South American countries
5,AME_N,Northern America
6,AME_N_OTH,Other Northern American countries
7,ASI,Asia
8,ASI_OTH,Other Asian countries (aggregate changing acco...
9,AT,Austria


In [43]:
nights_market_codes = set(
    source_nights_clean[
        "source_market_code"
    ].unique()
)

arrivals_market_codes = set(
    source_arrivals_clean[
        "source_market_code"
    ].unique()
)

print(
    "Nights categories:",
    len(nights_market_codes)
)

print(
    "Arrivals categories:",
    len(arrivals_market_codes)
)

print(
    "Same categories:",
    nights_market_codes
    == arrivals_market_codes
)

print(
    "Only in nights:",
    nights_market_codes
    - arrivals_market_codes
)

print(
    "Only in arrivals:",
    arrivals_market_codes
    - nights_market_codes
)

Nights categories: 63
Arrivals categories: 63
Same categories: True
Only in nights: set()
Only in arrivals: set()


## Source Market Classification

The Germany source-market datasets contain both individual countries and aggregated geographic categories.

These categories cannot be analyzed together as independent source markets because doing so would double count tourism demand. For example, individual European countries coexist with Europe, EU27, EFTA, and other European aggregates.

Source-market codes will therefore be classified into:

- `individual_country` - individually reported foreign countries
- `combined_market` - combined Switzerland and Liechtenstein market
- `domestic` - Germany or domestic-country categories
- `regional_aggregate` - continental and regional totals
- `eu_aggregate` - EU and intra-EU aggregates
- `world_aggregate` - worldwide totals or unallocated world observations

Only individual international countries and explicitly identified combined markets will be eligible for international source-market rankings.

In [44]:
regional_aggregate_codes = {
    "AFR",
    "AFR_OTH",
    "AME",
    "AME_C_S",
    "AME_C_S_OTH",
    "AME_N",
    "AME_N_OTH",
    "ASI",
    "ASI_OTH",
    "EFTA",
    "EUR",
    "EUR_OTH",
    "OCE",
    "OCE_OTH"
}

eu_aggregate_codes = {
    "EU27_2020_FOR",
    "INT_EU27_2007",
    "INT_EU27_2020",
    "INT_EU28"
}

domestic_codes = {
    "DE",
    "DOM"
}

world_aggregate_codes = {
    "WORLD",
    "WRL_NAL"
}

combined_market_codes = {
    "CH_LI"
}

In [45]:
def classify_source_market(code):

    if code in domestic_codes:
        return "domestic"

    elif code in eu_aggregate_codes:
        return "eu_aggregate"

    elif code in regional_aggregate_codes:
        return "regional_aggregate"

    elif code in world_aggregate_codes:
        return "world_aggregate"

    elif code in combined_market_codes:
        return "combined_market"

    else:
        return "individual_country"

In [46]:
source_market_reference[
    "market_type"
] = (
    source_market_reference[
        "source_market_code"
    ]
    .apply(
        classify_source_market
    )
)

source_market_reference

,source_market_code,source_market,market_type
0,AFR,Africa,regional_aggregate
1,AFR_OTH,Other African countries (aggregate changing ac...,regional_aggregate
2,AME,America,regional_aggregate
3,AME_C_S,Central and South America,regional_aggregate
4,AME_C_S_OTH,Other Central or South American countries,regional_aggregate
5,AME_N,Northern America,regional_aggregate
6,AME_N_OTH,Other Northern American countries,regional_aggregate
7,ASI,Asia,regional_aggregate
8,ASI_OTH,Other Asian countries (aggregate changing acco...,regional_aggregate
9,AT,Austria,individual_country


In [47]:
market_type_summary = (
    source_market_reference[
        "market_type"
    ]
    .value_counts()
    .rename_axis(
        "market_type"
    )
    .reset_index(
        name="market_count"
    )
)

market_type_summary

,market_type,market_count
0,individual_country,40
1,regional_aggregate,14
2,eu_aggregate,4
3,domestic,2
4,world_aggregate,2
5,combined_market,1


In [48]:
print(
    "Total categories:",
    len(source_market_reference)
)

print(
    "Classified categories:",
    market_type_summary[
        "market_count"
    ].sum()
)

print(
    "\nClassification totals:"
)

display(
    market_type_summary
)

Total categories: 63
Classified categories: 63

Classification totals:


,market_type,market_count
0,individual_country,40
1,regional_aggregate,14
2,eu_aggregate,4
3,domestic,2
4,world_aggregate,2
5,combined_market,1


In [49]:
non_individual_markets = (
    source_market_reference[
        source_market_reference[
            "market_type"
        ] != "individual_country"
    ]
    .sort_values(
        [
            "market_type",
            "source_market_code"
        ]
    )
)

non_individual_markets

,source_market_code,source_market,market_type
15,CH_LI,Switzerland and Liechtenstein,combined_market
19,DE,Germany,domestic
21,DOM,Domestic country,domestic
26,EU27_2020_FOR,EU27 countries (from 2020) except reporting co...,eu_aggregate
34,INT_EU27_2007,Intra-EU27 (2007-2013),eu_aggregate
35,INT_EU27_2020,Intra-EU27 (from 2020),eu_aggregate
36,INT_EU28,Intra-EU28 (2013-2020),eu_aggregate
0,AFR,Africa,regional_aggregate
1,AFR_OTH,Other African countries (aggregate changing ac...,regional_aggregate
2,AME,America,regional_aggregate


In [50]:
source_market_reference[
    "international_market"
] = (
    source_market_reference[
        "market_type"
    ]
    .isin(
        [
            "individual_country",
            "combined_market"
        ]
    )
)

In [51]:
source_market_reference[
    [
        "market_type",
        "international_market"
    ]
].value_counts()

market_type         international_market
individual_country  True                    40
regional_aggregate  False                   14
eu_aggregate        False                    4
domestic            False                    2
world_aggregate     False                    2
combined_market     True                     1
Name: count, dtype: int64

In [52]:
market_classification = (
    source_market_reference[
        [
            "source_market_code",
            "market_type",
            "international_market"
        ]
    ]
)

source_nights_clean = (
    source_nights_clean
    .merge(
        market_classification,
        on="source_market_code",
        how="left",
        validate="many_to_one"
    )
)

source_arrivals_clean = (
    source_arrivals_clean
    .merge(
        market_classification,
        on="source_market_code",
        how="left",
        validate="many_to_one"
    )
)

In [53]:
print(
    "Source nights rows:",
    len(source_nights_clean)
)

print(
    "Source arrivals rows:",
    len(source_arrivals_clean)
)

print(
    "Missing nights classifications:",
    source_nights_clean[
        "market_type"
    ].isna().sum()
)

print(
    "Missing arrivals classifications:",
    source_arrivals_clean[
        "market_type"
    ].isna().sum()
)

Source nights rows: 315
Source arrivals rows: 315
Missing nights classifications: 0
Missing arrivals classifications: 0


### Source Market Availability

Source-market arrivals and nights contain unavailable observations that must be distinguished from zero tourism demand.

Missing official observations will remain missing.

Availability will be evaluated by year and market type before the source-market datasets are used for rankings, growth calculations, or average length-of-stay analysis.

In [54]:
source_nights_missing_by_year = (
    source_nights_clean
    .groupby("year")
    .agg(
        total_records=(
            "source_market_code",
            "count"
        ),
        available_values=(
            "value_available",
            "sum"
        )
    )
    .reset_index()
)

source_nights_missing_by_year[
    "missing_values"
] = (
    source_nights_missing_by_year[
        "total_records"
    ]
    -
    source_nights_missing_by_year[
        "available_values"
    ]
)

source_nights_missing_by_year

,year,total_records,available_values,missing_values
0,2021,63,62,1
1,2022,63,62,1
2,2023,63,62,1
3,2024,63,62,1
4,2025,63,0,63


In [55]:
source_arrivals_missing_by_year = (
    source_arrivals_clean
    .groupby("year")
    .agg(
        total_records=(
            "source_market_code",
            "count"
        ),
        available_values=(
            "value_available",
            "sum"
        )
    )
    .reset_index()
)

source_arrivals_missing_by_year[
    "missing_values"
] = (
    source_arrivals_missing_by_year[
        "total_records"
    ]
    -
    source_arrivals_missing_by_year[
        "available_values"
    ]
)

source_arrivals_missing_by_year

,year,total_records,available_values,missing_values
0,2021,63,62,1
1,2022,63,62,1
2,2023,63,62,1
3,2024,63,62,1
4,2025,63,0,63


In [56]:
source_nights_missing = (
    source_nights_clean[
        ~source_nights_clean[
            "value_available"
        ]
    ][
        [
            "source_market_code",
            "source_market",
            "year",
            "market_type"
        ]
    ]
)

source_nights_missing.groupby(
    [
        "source_market_code",
        "source_market",
        "market_type"
    ]
).agg(
    missing_years=("year", "count"),
    first_missing_year=("year", "min"),
    last_missing_year=("year", "max")
).reset_index()

,source_market_code,source_market,market_type,missing_years,first_missing_year,last_missing_year
0,AFR,Africa,regional_aggregate,1,2025,2025
1,AFR_OTH,Other African countries (aggregate changing ac...,regional_aggregate,1,2025,2025
2,AME,America,regional_aggregate,1,2025,2025
3,AME_C_S,Central and South America,regional_aggregate,1,2025,2025
4,AME_C_S_OTH,Other Central or South American countries,regional_aggregate,1,2025,2025
5,AME_N,Northern America,regional_aggregate,1,2025,2025
6,AME_N_OTH,Other Northern American countries,regional_aggregate,1,2025,2025
7,ASI,Asia,regional_aggregate,1,2025,2025
8,ASI_OTH,Other Asian countries (aggregate changing acco...,regional_aggregate,1,2025,2025
9,AT,Austria,individual_country,1,2025,2025


In [57]:
source_arrivals_missing = (
    source_arrivals_clean[
        ~source_arrivals_clean[
            "value_available"
        ]
    ][
        [
            "source_market_code",
            "source_market",
            "year",
            "market_type"
        ]
    ]
)

source_arrivals_missing.groupby(
    [
        "source_market_code",
        "source_market",
        "market_type"
    ]
).agg(
    missing_years=("year", "count"),
    first_missing_year=("year", "min"),
    last_missing_year=("year", "max")
).reset_index()

,source_market_code,source_market,market_type,missing_years,first_missing_year,last_missing_year
0,AFR,Africa,regional_aggregate,1,2025,2025
1,AFR_OTH,Other African countries (aggregate changing ac...,regional_aggregate,1,2025,2025
2,AME,America,regional_aggregate,1,2025,2025
3,AME_C_S,Central and South America,regional_aggregate,1,2025,2025
4,AME_C_S_OTH,Other Central or South American countries,regional_aggregate,1,2025,2025
5,AME_N,Northern America,regional_aggregate,1,2025,2025
6,AME_N_OTH,Other Northern American countries,regional_aggregate,1,2025,2025
7,ASI,Asia,regional_aggregate,1,2025,2025
8,ASI_OTH,Other Asian countries (aggregate changing acco...,regional_aggregate,1,2025,2025
9,AT,Austria,individual_country,1,2025,2025


In [58]:
key_validation = pd.DataFrame({
    "dataset": [
        "Germany Nights",
        "EU27 Nights",
        "Germany Arrivals",
        "EU27 Arrivals",
        "Source Market Nights",
        "Source Market Arrivals",
        "Travel BoP"
    ],

    "duplicate_keys": [
        germany_nights_clean.duplicated(
            ["country_code", "period"]
        ).sum(),

        eu27_nights_clean.duplicated(
            ["country_code", "period"]
        ).sum(),

        germany_arrivals_clean.duplicated(
            ["country_code", "period"]
        ).sum(),

        eu27_arrivals_clean.duplicated(
            ["country_code", "period"]
        ).sum(),

        source_nights_clean.duplicated(
            ["source_market_code", "year"]
        ).sum(),

        source_arrivals_clean.duplicated(
            ["source_market_code", "year"]
        ).sum(),

        travel_bop_clean.duplicated(
            [
                "partner_code",
                "flow_code",
                "year"
            ]
        ).sum()
    ]
})

key_validation

,dataset,duplicate_keys
0,Germany Nights,0
1,EU27 Nights,0
2,Germany Arrivals,0
3,EU27 Arrivals,0
4,Source Market Nights,0
5,Source Market Arrivals,0
6,Travel BoP,0


In [59]:
range_validation = pd.DataFrame({
    "dataset": [
        "Germany Nights",
        "EU27 Nights",
        "Germany Arrivals",
        "EU27 Arrivals",
        "Source Market Nights",
        "Source Market Arrivals"
    ],

    "negative_values": [
        (germany_nights_clean[
            "foreign_nights"
        ] < 0).sum(),

        (eu27_nights_clean[
            "foreign_nights"
        ] < 0).sum(),

        (germany_arrivals_clean[
            "foreign_arrivals"
        ] < 0).sum(),

        (eu27_arrivals_clean[
            "foreign_arrivals"
        ] < 0).sum(),

        (source_nights_clean[
            "foreign_nights"
        ] < 0).sum(),

        (source_arrivals_clean[
            "foreign_arrivals"
        ] < 0).sum()
    ]
})

range_validation

,dataset,negative_values
0,Germany Nights,0
1,EU27 Nights,0
2,Germany Arrivals,0
3,EU27 Arrivals,0
4,Source Market Nights,0
5,Source Market Arrivals,0


In [60]:
travel_bop_clean.groupby(
    "flow_code"
)["travel_mio_eur"].agg(
    ["count", "min", "max"]
)

,count,min,max
flow_code,,,
BAL,10,-50469.0,-3556.0
CRE,10,5510.0,25711.0
DEB,10,9065.0,75502.0


In [61]:
period_validation = pd.DataFrame({
    "check": [
        "Monthly minimum year",
        "Monthly maximum year",
        "Minimum month",
        "Maximum month",
        "Source minimum year",
        "Source maximum year",
        "BoP minimum year",
        "BoP maximum year"
    ],

    "value": [
        eu27_nights_clean["year"].min(),
        eu27_nights_clean["year"].max(),
        eu27_nights_clean["month"].min(),
        eu27_nights_clean["month"].max(),
        source_nights_clean["year"].min(),
        source_nights_clean["year"].max(),
        travel_bop_clean["year"].min(),
        travel_bop_clean["year"].max()
    ]
})

period_validation

,check,value
0,Monthly minimum year,2021
1,Monthly maximum year,2026
2,Minimum month,1
3,Maximum month,12
4,Source minimum year,2021
5,Source maximum year,2025
6,BoP minimum year,2021
7,BoP maximum year,2025


## Data Cleaning Validation

The cleaned datasets passed the required structural and data-integrity checks.

Key findings from the cleaning process are:

- no duplicate analytical keys were identified
- country, period, year, month, source-market, partner, and flow identifiers are complete
- monthly year and month fields are internally consistent
- tourism arrivals and overnight stays contain no negative values
- missing official observations were preserved rather than replaced with zero or statistically imputed
- Eurostat estimated, low-reliability, confidential, and provisional observations were retained and explicitly identified
- historical EU27 annual completeness varies for a small number of country-years and is documented for later annual comparisons
- Germany has complete monthly arrivals and overnight stays for 2021–2025
- Germany's current 2026 comparison period is January–June
- all 27 EU countries have complete January–June 2026 overnight-stay data
- 26 EU countries have complete January–June 2026 arrival data, with Portugal incomplete for this period
- Germany source-market arrivals and overnight stays are analytically available through 2024
- 2025 source-market observations are unavailable and will not be imputed
- individual source markets have been separated from domestic, regional, EU, world, and combined-market categories
- Balance of Payments travel data are complete for 2021–2025, with 2025 observations identified as provisional

These rules establish the valid analytical periods and comparison populations that will be used in subsequent notebooks.

In [62]:
clean_datasets = {
    "Germany Nights": germany_nights_clean,
    "EU27 Nights": eu27_nights_clean,
    "Germany Arrivals": germany_arrivals_clean,
    "EU27 Arrivals": eu27_arrivals_clean,
    "Source Market Nights": source_nights_clean,
    "Source Market Arrivals": source_arrivals_clean,
    "Travel BoP": travel_bop_clean,
    "EU27 Reference": eu27_reference_clean
}

schema_summary = []

for name, df in clean_datasets.items():

    schema_summary.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": df.duplicated().sum()
    })

schema_summary = pd.DataFrame(
    schema_summary
)

schema_summary

,dataset,rows,columns,duplicate_rows
0,Germany Nights,67,11,0
1,EU27 Nights,1809,14,0
2,Germany Arrivals,67,11,0
3,EU27 Arrivals,1809,14,0
4,Source Market Nights,315,10,0
5,Source Market Arrivals,315,10,0
6,Travel BoP,30,10,0
7,EU27 Reference,27,2,0


In [63]:
for name, df in clean_datasets.items():

    print(f"\n{name}")
    print(df.columns.tolist())


Germany Nights
['country_code', 'country', 'period', 'foreign_nights', 'status', 'year', 'month', 'value_available', 'date', 'months_available', 'complete_12_month_year']

EU27 Nights
['country_code', 'country', 'period', 'foreign_nights', 'status', 'year', 'month', 'value_available', 'is_estimated', 'is_low_reliability', 'is_confidential', 'date', 'months_available', 'complete_12_month_year']

Germany Arrivals
['country_code', 'country', 'period', 'foreign_arrivals', 'status', 'year', 'month', 'value_available', 'date', 'months_available', 'complete_12_month_year']

EU27 Arrivals
['country_code', 'country', 'period', 'foreign_arrivals', 'status', 'year', 'month', 'value_available', 'is_estimated', 'is_low_reliability', 'is_confidential', 'date', 'months_available', 'complete_12_month_year']

Source Market Nights
['destination_code', 'destination', 'source_market_code', 'source_market', 'year', 'foreign_nights', 'status', 'value_available', 'market_type', 'international_market']

Sour

In [64]:
germany_nights_clean.to_csv(
    clean_path /
    "germany_foreign_nights_monthly_clean.csv",
    index=False
)

eu27_nights_clean.to_csv(
    clean_path /
    "eu27_foreign_nights_monthly_clean.csv",
    index=False
)

germany_arrivals_clean.to_csv(
    clean_path /
    "germany_foreign_arrivals_monthly_clean.csv",
    index=False
)

eu27_arrivals_clean.to_csv(
    clean_path /
    "eu27_foreign_arrivals_monthly_clean.csv",
    index=False
)

source_nights_clean.to_csv(
    clean_path /
    "germany_source_market_nights_annual_clean.csv",
    index=False
)

source_arrivals_clean.to_csv(
    clean_path /
    "germany_source_market_arrivals_annual_clean.csv",
    index=False
)

travel_bop_clean.to_csv(
    clean_path /
    "germany_travel_bop_annual_clean.csv",
    index=False
)

eu27_reference_clean.to_csv(
    clean_path /
    "eu27_country_reference_clean.csv",
    index=False
)

print("Clean datasets saved successfully.")

Clean datasets saved successfully.


In [65]:
source_market_reference.to_csv(
    clean_path /
    "source_market_reference_clean.csv",
    index=False
)

print(
    "Source market reference saved successfully."
)

Source market reference saved successfully.


In [66]:
clean_files = sorted(
    clean_path.glob("*.csv")
)

print(
    "Number of clean CSV files:",
    len(clean_files)
)

for file in clean_files:
    print(file.name)

Number of clean CSV files: 9
eu27_country_reference_clean.csv
eu27_foreign_arrivals_monthly_clean.csv
eu27_foreign_nights_monthly_clean.csv
germany_foreign_arrivals_monthly_clean.csv
germany_foreign_nights_monthly_clean.csv
germany_source_market_arrivals_annual_clean.csv
germany_source_market_nights_annual_clean.csv
germany_travel_bop_annual_clean.csv
source_market_reference_clean.csv


In [67]:
saved_file_validation = []

for file in clean_files:

    df_check = pd.read_csv(
        file
    )

    saved_file_validation.append({
        "file": file.name,
        "rows": len(df_check),
        "columns": len(df_check.columns)
    })

saved_file_validation = pd.DataFrame(
    saved_file_validation
)

saved_file_validation

,file,rows,columns
0,eu27_country_reference_clean.csv,27,2
1,eu27_foreign_arrivals_monthly_clean.csv,1809,14
2,eu27_foreign_nights_monthly_clean.csv,1809,14
3,germany_foreign_arrivals_monthly_clean.csv,67,11
4,germany_foreign_nights_monthly_clean.csv,67,11
5,germany_source_market_arrivals_annual_clean.csv,315,10
6,germany_source_market_nights_annual_clean.csv,315,10
7,germany_travel_bop_annual_clean.csv,30,10
8,source_market_reference_clean.csv,63,4


## Cleaning Outcome

The data cleaning stage is complete.

Nine cleaned datasets were created and successfully reloaded from disk without structural loss.

The cleaning process preserved official Eurostat observations, missing values, and quality-status information while adding analytical fields for:

- value availability
- observation quality
- annual completeness
- source-market classification
- international-market eligibility
- provisional Balance of Payments status

No statistical imputation was applied.

Historical and year-to-date completeness rules have been documented so that later aggregation, ranking, growth, and statistical analysis use only comparable observations.

The cleaned datasets are now ready for exploratory data analysis.